# remyelin pipeline, cropped region removed

This notebook reproduces the `demo-rml-local-figure.ipynb` pipeline, but first removes the previously-identified cropped region (see that notebook's `cropped` column / matching logic) from the raw data, **before** re-fitting NMF, so the whole downstream pipeline (NMF, factor regression, spatial stats, gene-wise analysis) is re-run on the pruned dataset rather than reusing any fit that included the cropped region.

**Important:** NMF factor indices are *not* guaranteed to line up with the original notebook's (e.g. `k=6`/`k=27` were case-associated there) since removing a chunk of tissue can reorganize the whole decomposition. `CASE_KS`/`CTL_KS` are left empty below — inspect the factor beta plot partway through this notebook and fill them in for this run before continuing to the case/control-specific plotting and gene-wise analysis cells.

In [ ]:
# ruff: noqa: F405
%cd /lustre/scratch126/gengen/teams_v2/marks/dp31/elia/spatialPeeler/demo

from _shared import pl_k_sP, remyelin_ds, run_deg_groups, run_nmf, run_proc_per_ds, viz_defs, comp_spat_stats  # isort:skip

from functools import partial
from pathlib import Path
from pickle import dump, load
from warnings import catch_warnings, simplefilter

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import uapl
import upsetplot as up
from matplotlib import pyplot as plt
from plotnine import *  # noqa: F403
from scipy import io

import spatialpeeler as sP
from spatialpeeler.diag import comp_sparsity, sweep_uns


In [ ]:
viz_defs(dpi=128)
RAND_SEED = 28
# define helper function for plotting
pl_help = partial(pl_k_sP, cond_col="Condition", sample_col="puck_id",
ctl_val="Saline", sp_p_size=0.3)
SP_FACT_PAT_STR = "sP_p_hat_k={k}"
SP_GENE_PAT_STR = "sp_gene_id_k={k}"

# NOTE: NMF is refit from scratch on the cropped-removed data below, so factor
# indices are NOT guaranteed to correspond to the same biological signatures as
# in demo-rml-local-figure.ipynb (there, k=6/27 were case-associated, k=1 control-
# associated). Inspect the factor beta plot further down in this notebook and
# update these once you've re-identified the case/control-specific factors.
CASE_KS = []
CTL_KS = []


In [ ]:
# load dataset (cropped region removed) + run NMF
if (pth := Path("../.cache/rml_cropremoved.pkl")).exists():
    with open(pth, "rb") as f:
        rml = load(f)
else:
    adatas = remyelin_ds(
        Path("../data/remyelin-slideseq"),
        # load only subset of interest
        lambda obs: (
            (obs["GoodQuality"] == "Yes")
            & (
                ((obs["Timepoint"].isin([3])) & (obs["Condition"] == "LPC"))
                | (obs["Timepoint"].isin([12, 18]) & (obs["Condition"] == "Saline"))
            )
        ),
    )
    for e in adatas.values():
        sc.pp.filter_cells(e, min_counts=100)

    run_proc_per_ds(adatas)

    rml = ad.concat(adatas, label="puck_id", index_unique="_")

    # --- remove the cropped region before any further processing ---
    base = "/lustre/scratch126/gengen/teams_v2/marks/dp31//SpatialPeeler/Data/Remyelin_Slide-seq/all_final_cropped_pucks_standardpipeline"
    cropped_obs_names = pd.read_csv(f"{base}/all_final_cropped_pucks_standardpipeline_barcodes.tsv", header=None)[0].values

    # cropped barcodes are "{puck_id}_{barcode}-N", rml's are "{barcode}-N_{puck_id}" (from ad.concat's index_unique="_")
    pattern = r"^(?P<puck>Puck_\d+_\d+)_(?P<barcode>[ACGTN]+-\d+)$"
    parsed = pd.Index(cropped_obs_names).str.extract(pattern)
    assert parsed.notna().all().all(), "some cropped obs_names didn't match the expected Puck_..._barcode-N pattern"
    cropped_barcodes = set(parsed["barcode"] + "_" + parsed["puck"])

    is_cropped = rml.obs_names.isin(cropped_barcodes)
    n_cropped = int(is_cropped.sum())
    print(f"removing {n_cropped} / {rml.n_obs} spots that fall in the cropped region ({n_cropped / rml.n_obs:.1%})")
    rml = rml[~is_cropped].copy()

    sc.pp.filter_genes(rml, min_cells=rml.n_obs // 500)

    rml.var["SYMBOL"] = (
        sc.queries.biomart_annotations("mmusculus", ["ensembl_gene_id", "external_gene_name"])
        .set_index("ensembl_gene_id")
        .merge(rml.var.index.to_series(), left_index=True, right_index=True, how="right")["external_gene_name"]
        .fillna(rml.var.index.to_series())
    )

    rml.obs["puck_id_cond"] = rml.obs["puck_id"].str.cat(rml.obs["Condition"], sep="_")
    rml.obs["Condition"] = rml.obs["Condition"].astype("category").cat.reorder_categories(["Saline", "LPC"])

    run_nmf(rml, 30, RAND_SEED)

    pth.parent.mkdir(exist_ok=True)
    with open(pth, "wb") as f:
        dump(rml, f)
print(rml)


before proceeding with spatialPeeler, we inspect our count matrix and NMF factors to assess their sparsity

In [ ]:
X_spars = rml.X.nnz / np.prod(rml.X.shape)  # ty:ignore[unresolved-attribute]
(
    comp_sparsity(rml.obsm["X_nmf"], cond=rml.obs["Condition"])
    >> ggplot()
    + aes(x="density", y="column", color="cond")
    + geom_point()
    + scale_y_discrete(limits=reversed)  # ty:ignore[invalid-argument-type]
    + labs(
        x="fraction of non-zero observations",
        y="NMF factor",
        title=f"fraction of non-zero observations in count matrix: {X_spars * 100:.02f}%",
    )
).show()


we see that the dataset is overall quite sparse (only ~1% of the data is non zero)
and this is reflected in the factors, which are also mostly quite sparse \
as such, we threshold our factor regression analysis to only include cells which are non-sparse for that factor

In [ ]:
# run factor-wise regression step
for k in range(rml.obsm["X_nmf"].shape[1]):
    sP.run_fact_reg(
        rml,
        "Condition",
        "Saline",
        k,
        SP_FACT_PAT_STR.format(k=k),
        X_key="X_nmf",
        filt=lambda adata: adata[adata.obsm["X_nmf"][:, k] > 0, :],
    )

print(rml)


we now assess our resulting regression outputs distributions \
we can extract and collate the model betas and other statistics using the sweep_uns function

In [ ]:
(
    sweep_uns(rml, SP_FACT_PAT_STR)
    >> ggplot()
    + aes(x="reorder(k, beta)", fill="-np.log(pval)", y="beta")
    + geom_col(color="black")
    + coord_flip()
    + labs(fill="-log(pval)", x="NMF factor")
    + scale_fill_cmap("turbo")
    + scale_y_symlog()
).show()


**stop here and inspect the plot above** -- identify which factors look case- (LPC) vs control- (Saline) associated for *this* re-fit, then go back up to the params cell and set `CASE_KS`/`CTL_KS` accordingly before continuing.

to explore the spatial properties of the resulting factor patterns, we can compute and plot
spatial statistics for each on a per sample basis (in this case Geary's C and Morans' I):

In [ ]:
rml.raw = None  # drop raw to avoid copying the full sparse matrix on every AnnData slice in comp_spat_stats


In [ ]:
sp_stat_df, sp_stat_pl = comp_spat_stats(rml, SP_FACT_PAT_STR, "puck_id", "Condition", "Saline")


In [ ]:
(sp_stat_pl & theme(figure_size=(16, 9))).show()  # ty:ignore[unsupported-operator]


we plot the spatial distributions for all case-specific factors:

In [ ]:
for k in CASE_KS:
    for p in pl_help(rml[rml.obsm["X_nmf"][:, k] > 0, :], k, sp_p_size=0.7).values():
        p.show()


we plot the spatial distributions for all control-specific factors:

In [ ]:
for k in CTL_KS:
    for p in pl_help(rml[rml.obsm["X_nmf"][:, k] > 0, :], k, sp_p_size=0.7).values():
        p.show()


to explore this further, we proceed w/ a gene-wise analysis for all case- and control-specific factors:

In [ ]:
ks_oi = CASE_KS + CTL_KS

for k in ks_oi:
    sP.run_gene_reg(rml, SP_FACT_PAT_STR.format(k=k), SP_GENE_PAT_STR.format(k=k), layer="counts", run_reg=False)


In [ ]:
if ks_oi:
    phat_genes = pd.concat(
        [
            rml.varm[SP_GENE_PAT_STR.format(k=k)].assign(SYMBOL=rml.var["SYMBOL"], k=k).sort_values(
                "corr", ascending=False
            )
            for k in ks_oi
        ]
    ).reset_index(names="gene")
else:
    print("ks_oi is empty (CASE_KS/CTL_KS not yet set) -- skipping gene-wise analysis")
    phat_genes = pd.DataFrame(columns=["gene", "corr", "wcorr", "SYMBOL", "k"])
phat_genes


In [ ]:
out_path = Path("../.cache/top_phat_genes_rml_cropremoved.csv")
out_path.parent.mkdir(exist_ok=True)
phat_genes.to_csv(out_path, index=False)
print(f"saved {len(phat_genes)} rows to {out_path}")


we rank genes by their correlation with each factor's p̂ pattern, and list the top genes for each case-specific factor:

In [ ]:
for k in CASE_KS:
    print(f"k={k}")
    display(phat_genes.loc[phat_genes["k"] == k].drop(columns="k").head(25))


and for each control-specific factor:

In [ ]:
for k in CTL_KS:
    print(f"k={k}")
    display(phat_genes.loc[phat_genes["k"] == k].drop(columns="k").head(25))


In [ ]:
for k in ks_oi:
    (
        uapl.PlotAdata(rml)
        + aes(
            x=uapl.varm("X_nmf", k),
            y=uapl.varm(SP_GENE_PAT_STR.format(k=k), "corr"),
            label=np.where(
                np.argsort(np.argsort(-np.abs(np.asarray(rml.varm[SP_GENE_PAT_STR.format(k=k)]["corr"])))) < 8,
                rml.var["SYMBOL"],
                None,  # ty:ignore[invalid-argument-type]
            ),
        )
        + geom_point(size=1.5, shape="o", stroke=0)
        + geom_label(adjust_text={}, na_rm=True)
        + labs(
            colour="-log(pval)",
            x=f"NMF[{k}]",
            y="pearson correlation",
            title=f"NMF loading / p\u0302-gene correlation relationship for k={k}",
        )
    ).show()
